# SPIN Baseline Comparison

**What this does:**
1. Downloads Stage 4 results (image IDs + baseline captions) directly from GitHub
2. Generates SPIN captions on those exact same 400 images
3. Scores Baseline vs SPIN with CHAIR + bootstrap CIs

**Runtime:** A100 · **Time:** ~30 min · no LoRA loaded

## 0. Install (run once, restart runtime, then skip)

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!pip install -q 'numpy==1.26.4'
!pip install -q 'transformers>=4.47' 'accelerate>=0.33' 'tokenizers>=0.21'
!pip install -q peft bitsandbytes safetensors 'torchao>=0.16.0'
!pip install -q pillow tqdm spacy sentencepiece
!python -m spacy download en_core_web_sm -q
print('Done. Runtime -> Restart session, then skip this cell.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 129.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2

## 1. Imports + GPU check

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json, gc, time, urllib.request
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    raise RuntimeError('No GPU — Runtime -> Change runtime type -> A100 GPU')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/llava_hallucination_heads')
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / 'cache').mkdir(exist_ok=True)
(DRIVE / 'results').mkdir(exist_ok=True)

LOCAL = Path('/content/spin_work')
LOCAL.mkdir(exist_ok=True)
(LOCAL / 'images').mkdir(exist_ok=True)

GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Mounted at /content/drive


## 2. Download Stage 4 results from GitHub

In [2]:
S4_URL   = 'https://raw.githubusercontent.com/armaansandhu26/causal-grounding-lora/refs/heads/master/results/stage4_400img_results.json'
S4_LOCAL = LOCAL / 'stage4_400img_results.json'

if not S4_LOCAL.exists():
    print('Downloading Stage 4 results from GitHub...')
    urllib.request.urlretrieve(S4_URL, str(S4_LOCAL))
    print('Done.')

with open(S4_LOCAL) as f:
    s4 = json.load(f)

# Extract image IDs, GT objects, and baseline captions
eval_images     = [r['img_id']        for r in s4['eval_captions']]
eval_gt_objects = [set(r['gt'])       for r in s4['eval_captions']]
baseline_by_id  = {r['img_id']: r['captions']['baseline']
                   for r in s4['eval_captions']}

print(f'Images: {len(eval_images)}')
print(f'Baseline captions: {len(baseline_by_id)}')
print(f'Stage 4 CHAIR results (from repo):')
for cond, v in s4['chair'].items():
    print(f'  {cond:10s}  CHAIRs={v["CHAIRs"]:.4f}  CHAIRi={v["CHAIRi"]:.4f}')

Done.
Images: 400
Baseline captions: 400
Stage 4 CHAIR results (from repo):
  baseline    CHAIRs=0.3625  CHAIRi=0.1375
  stage2      CHAIRs=0.0725  CHAIRi=0.0389
  stage3      CHAIRs=0.3500  CHAIRi=0.1344
  stage4      CHAIRs=0.0700  CHAIRi=0.0391


## 3. Download the 400 eval images

In [3]:
IMG_DIR = LOCAL / 'images'
img_id_to_path = {}
to_dl = []

for img_id in eval_images:
    fname = f'COCO_val2014_{img_id:012d}.jpg'
    p = IMG_DIR / fname
    img_id_to_path[img_id] = str(p)
    if not p.exists():
        to_dl.append((img_id, fname, p))

if to_dl:
    print(f'Downloading {len(to_dl)} images...')
    for img_id, fname, p in tqdm(to_dl, desc='Images'):
        try:
            urllib.request.urlretrieve(
                f'http://images.cocodataset.org/val2014/{fname}', str(p))
        except Exception as e:
            print(f'  Failed {img_id}: {e}')
else:
    print('All images already downloaded.')

print(f'Ready: {len(eval_images)} images')

Images:   0%|          | 0/400 [00:00<?, ?it/s]

Ready: 400 images


## 4. Load LLaVA-1.5-7B (base model only, no LoRA)

In [4]:
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID  = 'llava-hf/llava-1.5-7b-hf'
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation='eager',
    device_map={'': 0},
)
model.eval()

# Model constants
text_cfg       = model.config.text_config
NUM_LAYERS     = text_cfg.num_hidden_layers       # 32
NUM_HEADS      = text_cfg.num_attention_heads      # 32
HEAD_DIM       = text_cfg.hidden_size // NUM_HEADS
IMAGE_TOKEN_ID = model.config.image_token_index
vision_cfg     = model.config.vision_config
NUM_IMG_TOKENS = (vision_cfg.image_size // vision_cfg.patch_size) ** 2  # 576
PROMPT         = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

# Correct decoder layers path for this transformers version
DECODER_LAYERS = model.model.language_model.layers

print(f'Model loaded.  Layers={NUM_LAYERS}, Heads={NUM_HEADS}, HeadDim={HEAD_DIM}')
print(f'Decoder layers: {len(DECODER_LAYERS)}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded.  Layers=32, Heads=32, HeadDim=128
Decoder layers: 32
VRAM: 14.13 GB


## 5. SPIN generation function

In [5]:
def get_visual_token_span(input_ids):
    ids  = input_ids[0]
    mask = (ids == IMAGE_TOKEN_ID)
    pos  = mask.nonzero(as_tuple=True)[0]
    n_ph = int(mask.sum().item())
    if n_ph >= NUM_IMG_TOKENS:
        return int(pos[0].item()), int(pos[-1].item()) + 1
    return int(pos[0].item()), int(pos[0].item()) + NUM_IMG_TOKENS


class SPINHook:
    """Suppresses image-inattentive heads at each decode step."""
    def __init__(self, img_start, img_end, n_keep, scale, num_heads, head_dim):
        self.img_start = img_start
        self.img_end   = img_end
        self.n_keep    = n_keep
        self.scale     = scale
        self.num_heads = num_heads
        self.head_dim  = head_dim
        self.handle    = None

    def __call__(self, module, input, output):
        attn_output  = output[0]   # [1, seq_len, hidden]
        attn_weights = output[1]   # [1, H, Q, K] or None
        if attn_weights is None:
            return output
        # Image attention mass for the last query position, per head
        img_attn = attn_weights[:, :, -1, self.img_start:self.img_end].sum(dim=-1)  # [1, H]
        _, sorted_idx = img_attn.sort(dim=-1, descending=True)
        suppress_idx  = sorted_idx[:, self.n_keep:]
        if suppress_idx.numel() == 0:
            return output
        reshaped = attn_output.view(1, -1, self.num_heads, self.head_dim)
        for h_idx in suppress_idx[0]:
            reshaped[:, -1:, h_idx, :] *= self.scale
        return (reshaped.view_as(attn_output),) + output[1:]

    def register(self, module):
        self.handle = module.register_forward_hook(self)
    def remove(self):
        if self.handle: self.handle.remove()


@torch.no_grad()
def gen_spin(model_obj, image_path, decoder_layers,
             start_layer=0, end_layer=32,
             keep_ratio=0.95, scale=0.0,
             max_new_tokens=80):
    """
    SPIN (Sarkar et al., EMNLP 2025): suppress image-inattentive heads.
    keep_ratio=0.95 -> keep top 95% of heads by image attention, suppress bottom 5%.
    scale=0.0 -> zero out suppressed heads' contribution.
    """
    img    = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT, images=img,
                       return_tensors='pt').to(device, torch.float16)
    inputs['input_ids']      = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    img_start, img_end = get_visual_token_span(inputs['input_ids'])
    eos_id = processor.tokenizer.eos_token_id
    n_keep = max(1, int(np.ceil(keep_ratio * NUM_HEADS)))

    # Register hooks on self_attn of each decoder layer
    hooks = []
    for l in range(start_layer, min(end_layer, len(decoder_layers))):
        hook = SPINHook(img_start, img_end, n_keep, scale, NUM_HEADS, HEAD_DIM)
        hook.register(decoder_layers[l].self_attn)
        hooks.append(hook)

    past_kv = None
    cur_ids = inputs['input_ids']
    cur_msk = inputs['attention_mask']
    generated = []

    try:
        for _ in range(max_new_tokens):
            kw = dict(input_ids=cur_ids, attention_mask=cur_msk,
                      use_cache=True, past_key_values=past_kv,
                      output_attentions=True, return_dict=True)
            if past_kv is None:
                kw['pixel_values'] = inputs['pixel_values']
            out = model_obj(**kw)

            next_id = int(out.logits[:, -1, :].float().argmax(dim=-1).item())
            generated.append(next_id)
            if next_id == eos_id:
                break

            past_kv = out.past_key_values
            cur_ids = torch.tensor([[next_id]], dtype=torch.long, device=device)
            cur_msk = torch.cat([cur_msk,
                                 torch.ones(1, 1, dtype=torch.long, device=device)], dim=1)
    finally:
        for h in hooks:
            h.remove()

    return processor.tokenizer.decode(generated, skip_special_tokens=True)


# Sanity test
test_path = img_id_to_path[eval_images[0]]
print('Sanity test...')
cap = gen_spin(model, test_path, DECODER_LAYERS)
torch.cuda.empty_cache()
print(f'  SPIN: {cap[:120]}')
print(f'  Baseline (from Stage 4): {baseline_by_id[eval_images[0]][:120]}')
print(f'VRAM after test: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('OK.')

Sanity test...
  SPIN: The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant is positioned near 
  Baseline (from Stage 4): The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant is prominently plac
VRAM after test: 14.14 GB
OK.


## 6. Generate SPIN captions for all 400 images

Saves to Drive every 10 images. Resume-safe — re-run this cell if it crashes.

In [6]:
SPIN_CKPT = DRIVE / 'cache' / 'spin_captions_s4ids.json'

if SPIN_CKPT.exists():
    with open(SPIN_CKPT) as f:
        spin_caps = json.load(f)
    done_ids = {r['img_id'] for r in spin_caps}
    print(f'Resumed: {len(done_ids)}/{len(eval_images)} done')
else:
    spin_caps = []
    done_ids  = set()

t0         = time.time()
fail_count = 0

for img_id, gt_set in tqdm(zip(eval_images, eval_gt_objects),
                            total=len(eval_images), desc='SPIN'):
    if img_id in done_ids:
        continue

    caption = ''
    try:
        caption = gen_spin(model, img_id_to_path[img_id], DECODER_LAYERS)
    except Exception as e:
        fail_count += 1
        print(f'  {img_id}: {e}')
        torch.cuda.empty_cache()
        gc.collect()

    spin_caps.append({'img_id': img_id, 'gt': list(gt_set), 'caption': caption})
    done_ids.add(img_id)

    if len(spin_caps) % 10 == 0:
        with open(SPIN_CKPT, 'w') as f:
            json.dump(spin_caps, f)
        torch.cuda.empty_cache()

with open(SPIN_CKPT, 'w') as f:
    json.dump(spin_caps, f)

print(f'Done: {len(spin_caps)} images in {(time.time()-t0)/60:.1f} min')
print(f'Failures: {fail_count}')
print(f'Saved: {SPIN_CKPT}')

SPIN:   0%|          | 0/400 [00:00<?, ?it/s]

Done: 400 images in 27.1 min
Failures: 0
Saved: /content/drive/MyDrive/llava_hallucination_heads/cache/spin_captions_s4ids.json


---
## Phase 2: Scoring (no GPU needed)
---

## 7. COCO vocab + CHAIR scorer

In [7]:
import spacy
nlp = spacy.load('en_core_web_sm')

COCO_SYNONYMS = {
    'person':       ['man','woman','people','boy','girl','child','guy','lady',
                     'kid','baby','player','rider','skier','surfer','snowboarder'],
    'car':          ['vehicle','automobile','sedan','suv'],
    'dog':          ['puppy','dogs'],
    'cat':          ['kitten','cats'],
    'tv':           ['television','monitor','screen'],
    'couch':        ['sofa'],
    'cell phone':   ['phone','cellphone','smartphone'],
    'dining table': ['table','desk'],
    'wine glass':   ['glass'],
    'bicycle':      ['bike'],
    'motorcycle':   ['motorbike'],
    'airplane':     ['plane','jet'],
    'potted plant': ['plant'],
    'laptop':       ['computer'],
    'refrigerator': ['fridge'],
    'truck':        ['lorry'],
    'boat':         ['ship','sailboat'],
    'fire hydrant': ['hydrant'],
    'hot dog':      ['hotdog'],
    'traffic light':['stoplight'],
    'sports ball':  ['ball','football','soccer ball','basketball'],
    'baseball bat': ['bat'],
    'tennis racket':['racket','racquet'],
}
MULTIWORD = {
    'hydrant':'fire hydrant', 'hotdog':'hot dog', 'stoplight':'traffic light',
    'bat':'baseball bat',     'racket':'tennis racket', 'racquet':'tennis racket',
}
ALL_COCO = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors',
    'teddy bear','hair drier','toothbrush',
]
OBJECT_VOCAB = set(ALL_COCO)
for syns in COCO_SYNONYMS.values(): OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD.keys())


def find_content_words(caption, gt_objects):
    gt_norm     = {o.lower() for o in gt_objects}
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm: expanded_gt.update(syns)
    for alias, canonical in MULTIWORD.items():
        if canonical in gt_norm: expanded_gt.add(alias)
    doc = nlp(caption)
    obj_words, hall_words = [], []
    for tok in doc:
        w = tok.text.lower().strip()
        if tok.pos_ not in ('NOUN', 'PROPN') or len(w) < 2: continue
        canonical = MULTIWORD.get(w, w)
        if w in OBJECT_VOCAB or canonical in OBJECT_VOCAB:
            obj_words.append(w)
            if w not in expanded_gt and canonical not in expanded_gt:
                hall_words.append(w)
    return obj_words, hall_words


def score_chair(captions_gt):
    chairs_list, chairi_list = [], []
    for cap, gt in captions_gt:
        if not cap: continue
        obj_w, hall_w = find_content_words(cap, gt)
        chairs_list.append(1 if hall_w else 0)
        chairi_list.append(len(hall_w) / max(len(obj_w), 1))
    return (float(np.mean(chairs_list)) if chairs_list else 0.0,
            float(np.mean(chairi_list)) if chairi_list else 0.0)


print('CHAIR scorer ready.')

CHAIR scorer ready.


## 8. Results: Baseline vs SPIN + Bootstrap CIs

In [8]:
# Load SPIN captions
with open(DRIVE / 'cache' / 'spin_captions_s4ids.json') as f:
    spin_caps = json.load(f)

# Load Stage 4 results (baseline captions + GT)
with open(LOCAL / 'stage4_400img_results.json') as f:
    s4 = json.load(f)
baseline_by_id = {r['img_id']: {'caption': r['captions']['baseline'],
                                  'gt':      set(r['gt'])}
                  for r in s4['eval_captions']}

# Build paired records (should be 400/400)
paired = []
for r in spin_caps:
    iid = r['img_id']
    if iid not in baseline_by_id:
        continue
    paired.append({
        'img_id':       iid,
        'gt':           baseline_by_id[iid]['gt'],
        'baseline_cap': baseline_by_id[iid]['caption'],
        'spin_cap':     r['caption'],
    })

print(f'Paired images: {len(paired)} / {len(eval_images)}')
missing = len(eval_images) - len(paired)
if missing: print(f'  Warning: {missing} images not matched')

# Show 3 examples
print('\n--- Sample outputs ---')
for p in paired[:3]:
    print(f'  [{p["img_id"]}] GT: {sorted(p["gt"])[:4]}')
    print(f'    Baseline: {p["baseline_cap"][:100]}')
    print(f'    SPIN:     {p["spin_cap"][:100]}')
    print()


# Bootstrap CI
def bootstrap_ci(values, n_boot=2000, seed=42):
    rng  = np.random.RandomState(seed)
    arr  = np.array(values)
    boot = [arr[rng.randint(0, len(arr), len(arr))].mean() for _ in range(n_boot)]
    return float(arr.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


# Score
results = {}
for method, cap_key in [('Baseline', 'baseline_cap'), ('SPIN (Sarkar+25)', 'spin_cap')]:
    chair_pairs = [(p[cap_key], p['gt']) for p in paired if p[cap_key]]
    chairs, chairi = score_chair(chair_pairs)
    lengths = [len(p[cap_key].split()) for p in paired if p[cap_key]]

    per_img_s, per_img_i = [], []
    for p in paired:
        cap = p[cap_key]
        if not cap: continue
        obj_w, hall_w = find_content_words(cap, p['gt'])
        per_img_s.append(1 if hall_w else 0)
        per_img_i.append(len(hall_w) / max(len(obj_w), 1))

    cs_m, cs_lo, cs_hi = bootstrap_ci(per_img_s)
    ci_m, ci_lo, ci_hi = bootstrap_ci(per_img_i)

    results[method] = {
        'CHAIRs': chairs, 'CHAIRs_CI': (cs_lo, cs_hi),
        'CHAIRi': chairi, 'CHAIRi_CI': (ci_lo, ci_hi),
        'avg_len': float(np.mean(lengths)),
        'n': len(chair_pairs),
    }

# Print table
print('='*80)
print(f'{"Method":<22} {"CHAIRs":>8} {"95% CI":>18}  '
      f'{"CHAIRi":>8} {"95% CI":>18} {"AvgLen":>7}')
print('-'*80)

base_cs = results['Baseline']['CHAIRs']
for method, r in results.items():
    cs_lo, cs_hi = r['CHAIRs_CI']
    ci_lo, ci_hi = r['CHAIRi_CI']
    delta = '' if method == 'Baseline' else \
            f'  ({(base_cs - r["CHAIRs"]) / base_cs * 100:+.1f}%)'
    print(f'{method:<22} {r["CHAIRs"]:>8.3f} [{cs_lo:.3f}, {cs_hi:.3f}]  '
          f'{r["CHAIRi"]:>8.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {r["avg_len"]:>7.1f}{delta}')

print('='*80)
print(f'\nn = {len(paired)} paired images')

# Also print Stage 4 numbers from the repo for reference
print('\n--- Stage 4 repo numbers (for reference) ---')
for cond, v in s4['chair'].items():
    print(f'  {cond:10s}  CHAIRs={v["CHAIRs"]:.4f}  CHAIRi={v["CHAIRi"]:.4f}')

Paired images: 400 / 400

--- Sample outputs ---
  [293474] GT: ['book', 'fire hydrant']
    Baseline: The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant
    SPIN:     The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant

  [465878] GT: ['person', 'surfboard']
    Baseline: The image captures a man skillfully riding a surfboard on a wave in the ocean. He is positioned in t
    SPIN:     The image captures a man skillfully riding a wave on a surfboard in the ocean. He is positioned in t

  [419401] GT: ['bicycle', 'train']
    Baseline: The image features a red train with a bicycle symbol on the side. The train is parked at a station, 
    SPIN:     The image features a red train with a bicycle symbol on the side of it. The train is parked, and the

Method                   CHAIRs             95% CI    CHAIRi             95% CI  AvgLen
-------------------------------------------------------

## 9. Save results

In [9]:
out = {
    'n_paired': len(paired),
    'spin_config': {'start_layer':0, 'end_layer':32, 'keep_ratio':0.95, 'scale':0.0},
    'results': results,
}
# Make tuples JSON serializable
for m in out['results']:
    out['results'][m]['CHAIRs_CI'] = list(out['results'][m]['CHAIRs_CI'])
    out['results'][m]['CHAIRi_CI'] = list(out['results'][m]['CHAIRi_CI'])

out_path = DRIVE / 'results' / 'stage5_spin_comparison.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {out_path}')

Saved: /content/drive/MyDrive/llava_hallucination_heads/results/stage5_spin_comparison.json
